# 06 Autoformer-lite Forecast

`autoformer_lite` は厳密なAutoformer本体ではなく、Autoformer-inspired / decomposition Transformer baseline である。moving average decomposition で trend と seasonal/residual に分け、過去24か月のwindowから次の1か月を予測する軽量モデルとして扱う。

今回は fixed B のみ、外生変数なし、単変量 `y = log(number_parcels)` を使う。予測後は `exp()` で `number_parcels` スケールに戻し、既存の forecast evaluation 基盤で評価する。test期間の実績値は再帰予測の入力に使わない。

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display


PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name != "Transport_amount_project" and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_connected_parcel_data
from src.forecasting.autoformer_lite import fit_autoformer_lite, forecast_autoformer_lite
from src.forecasting.evaluation import evaluate_forecasts
from src.forecasting.splits import make_fixed_split_b


DATA_PATH = PROJECT_ROOT / "data" / "processed" / "parcel_volume_connected.csv"
FORECAST_DIR = PROJECT_ROOT / "output" / "forecasts"
PREDICTIONS_DIR = FORECAST_DIR / "predictions"
METRICS_DIR = FORECAST_DIR / "metrics"
FIGURES_DIR = FORECAST_DIR / "figures"

for path in [PREDICTIONS_DIR, METRICS_DIR, FIGURES_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data path:", DATA_PATH)
print("Forecast output:", FORECAST_DIR)

## 1. データ読み込みと fixed B split

接続済みデータを読み込み、fixed B の train/test を作成する。今回は fixed A は扱わない。

In [ ]:
df = load_connected_parcel_data(str(DATA_PATH))
split = make_fixed_split_b(df)
train = split["train"]
test = split["test"]

split_summary = pd.DataFrame(
    [
        {
            "split": split["split"],
            "train_start": train.index.min().date(),
            "train_end": train.index.max().date(),
            "train_rows": len(train),
            "test_start": test.index.min().date(),
            "test_end": test.index.max().date(),
            "test_rows": len(test),
        }
    ]
)
display(split_summary)

## 2. autoformer_lite の学習

train の `y` を平均・標準偏差で標準化し、lookback=24 の教師ありwindowを作る。モデルは小さな TransformerEncoder で、CPUでも短時間で終わる設定にしている。

In [ ]:
bundle = fit_autoformer_lite(
    train,
    target_col="y",
    lookback=24,
    max_epochs=100,
    random_seed=42,
    hidden_size=32,
    n_heads=2,
    n_layers=1,
    batch_size=32,
    learning_rate=1e-3,
    patience=10,
)

fit_summary = pd.DataFrame([bundle.config])
display(fit_summary)
display(bundle.train_history.tail(10))

## 3. fixed B の再帰予測

1か月先を予測し、その予測値を次の入力履歴に追加する。test期間の実績値は予測入力に使わない。

In [ ]:
forecast_df = forecast_autoformer_lite(bundle, test, split="fixed_b")
display(forecast_df.head())
display(forecast_df.tail())

## 4. 評価指標と既存モデルとの比較

評価は `number_parcels` の原系列スケールで行う。既存の `model_comparison_fixed_splits.csv` があれば fixed B の既存モデルと並べて確認する。

In [ ]:
autoformer_lite_metrics = evaluate_forecasts(
    forecast_df,
    y_train=train["number_parcels"],
)
display(autoformer_lite_metrics)

comparison_path = METRICS_DIR / "model_comparison_fixed_splits.csv"
if comparison_path.exists():
    comparison = pd.read_csv(comparison_path)
    fixed_b_comparison = comparison[comparison["split"] == "fixed_b"].copy()
    fixed_b_comparison = fixed_b_comparison[
        ["split", "model", "forecast_type", "information_set", "rmse", "mae", "mape", "mase"]
    ]
    autoformer_row = autoformer_lite_metrics.copy()
    autoformer_row["information_set"] = "unconditional"
    fixed_b_with_autoformer = pd.concat(
        [
            fixed_b_comparison,
            autoformer_row[["split", "model", "forecast_type", "information_set", "rmse", "mae", "mape", "mase"]],
        ],
        ignore_index=True,
    )
    display(fixed_b_with_autoformer.sort_values("rmse"))
else:
    print("model_comparison_fixed_splits.csv not found; skipping comparison table.")

## 5. 保存

予測結果、metrics、予測図を `output/forecasts/` 配下に保存する。論文用の `output/tables/` と `output/figures/` は使わない。

In [ ]:
forecast_df.to_csv(PREDICTIONS_DIR / "fixed_b_autoformer_lite.csv", index=False)
autoformer_lite_metrics.to_csv(METRICS_DIR / "autoformer_lite_metrics.csv", index=False)

fig, ax = plt.subplots(figsize=(10.5, 5.5))
train_tail = train.tail(24)
ax.plot(train_tail.index, train_tail["number_parcels"], color="0.55", linewidth=1.2, label="train actual tail")
ax.plot(test.index, test["number_parcels"], color="black", linewidth=1.6, label="test actual")
ax.plot(forecast_df["date"], forecast_df["y_pred"], marker="o", linewidth=1.2, label="autoformer_lite")
ax.axvline(train.index.max(), color="0.2", linestyle=":", linewidth=1.0)
ax.set_title("fixed_b: autoformer_lite forecast comparison")
ax.set_xlabel("Date")
ax.set_ylabel("number_parcels")
ax.grid(True, color="0.85", linewidth=0.8)
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "fixed_b_autoformer_lite_forecast.png", dpi=300, bbox_inches="tight")
plt.close(fig)

print("Saved autoformer_lite outputs.")

## 6. 読み取りメモ

`autoformer_lite` は小標本月次データで動かすための軽量な実験baselineであり、完全なAutoformer実装ではない。外生変数なしの unconditional forecast なので、SARIMAX/SSM conditional forecast とは情報条件が異なる。

学習曲線では train loss と validation loss の差を見る。validation loss が早めに悪化し、train loss だけ下がり続ける場合は overfitting の兆候として扱う。